# CA2

Link to the video for this CA ([HERE](https://nmbu.cloud.panopto.eu/Panopto/Pages/Viewer.aspx?id=67574a42-96b9-4946-aaa3-b4c801026e20))

## Context

Your department leader was happy with what you delivered in CA1. Now you got an email from her that says the following: "I have been thinking about your assignment and I need you to go a bit further with the hyperparameter optimisation process. Based on what you have done already, please also do the following":

- Instead of doing only a repeated stratified K-fold cross validation (as implemented in CA1), implement a nested cross-validation, where OptunaSearchCV is carried out in the inner loop
- In the outer loop use a repeated stratified KFold cross validation with 4 splits and 5 repeats
- In the inner loop use a repeated stratified KFold cross validation with 4 splits and 25 repeats
- Use 50 trials for the optimisation process
- For each outer fold compute the following performance metrics:
    - MCC
    - Scaled MCC (ranging from 0 to 1 instead of -1 to 1 as in MCC)
    - AUC
    - F1 pos (F1 score for the positive labels)
    - F1 neg (F1 score for the negative labels; you can compute this by flipping the labels)
    - Accuracy
    - Precision
    - Recall
- Construct a dataframe named named "metrics_df_outer" that holds the performance metric results for each outer fold. Add also columns for best C and best l1_ratio for each fold. (see example below in Figure 1)
- Compute average of the performance metrics and their standard deviation across the outer folds. Store the results in a dataframe named "metrics_df" (see example below in Figure 2)
- Your results will likely not be identical to those in the figures (because Optuna may not find exactly the same hyperparameter values for each of you), but they should be quite close
- Measure and report the time it takes to complete the optimisation process

## Imports

In [2]:
import pandas as pd
import time

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import matthews_corrcoef, make_scorer, roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

from optuna.integration import OptunaSearchCV
from optuna.distributions import FloatDistribution

c:\Users\milad\Documents\Progging\Machine-Learning-Deep-Learning\DAT350\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\milad\Documents\Progging\Machine-Learning-Deep-Learning\DAT350\.venv\Lib\site-packages\optuna\integration\sklearn.py:14: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  optuna_warn(f"{msg} Use `optuna_integration.sklearn` instead.", FutureWarning)


## Loading Data

In [3]:
df = pd.read_csv('hepatic_data.csv')
df.describe()

,Age,Gender,BMI,AlcoholConsumption,Smoking,GeneticRisk,PhysicalActivity,Diabetes,Hypertension,LiverFunctionTest,Diagnosis
count,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000,1700.000000
mean,50.394118,0.504118,27.699801,9.832309,0.291765,0.521765,5.000993,0.142353,0.154706,59.863867,0.550588
std,17.641915,0.500130,7.210400,5.757472,0.454708,0.666262,2.847074,0.349515,0.361730,22.996262,0.497581
min,20.000000,0.000000,15.004710,0.003731,0.000000,0.000000,0.001852,0.000000,0.000000,20.019254,0.000000
25%,35.000000,0.000000,21.455414,4.841811,0.000000,0.000000,2.622121,0.000000,0.000000,40.024216,0.000000
50%,51.000000,1.000000,27.925367,9.828195,0.000000,0.000000,5.022883,0.000000,0.000000,59.513146,1.000000
75%,66.000000,1.000000,33.957668,14.871671,1.000000,1.000000,7.401642,0.000000,0.000000,79.428755,1.000000
max,80.000000,1.000000,39.992845,19.952456,1.000000,2.000000,9.994964,1.000000,1.000000,99.991413,1.000000


## Modelling

In [4]:
# Splitting the data to features and target
X = df.drop("Diagnosis", axis = 1)
y = df["Diagnosis"]

In [ ]:
# Creating the parameters for the hyperparameter
parameters = {
    "classifier__C": FloatDistribution(1e-10, 1e+10, log = True),
    "classifier__l1_ratio": FloatDistribution(0, 1)
}

# Creating the scorer for the optimization
mcc_scorer = make_scorer(matthews_corrcoef)

In [7]:
# Creatinng the outer and inner cv
outer_cv = RepeatedStratifiedKFold(
    n_splits=4,
    n_repeats=5,
    random_state=42
)

inner_cv = RepeatedStratifiedKFold(
    n_splits=4,
    n_repeats=25,
    random_state=42
)

In [ ]:
scores_outer = []

# Staring time
start_time = time.perf_counter()

for fold, (train_i, test_i) in enumerate(outer_cv.split(X, y), 1):
    X_train, X_test = X.iloc[train_i], X.iloc[test_i]
    y_train, y_test = y.iloc[train_i], y.iloc[test_i]

    # Creating the pipeline with a scaler and a logistic regression with elasticnet penalty
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            penalty = "elasticnet",
            solver = "saga",
            max_iter = 10000
        ))
    ])

    optuna_search = OptunaSearchCV(
        estimator = pipeline,
        param_distributions = parameters,
        cv = inner_cv,
        scoring = mcc_scorer,
        n_trials = 50,
        n_jobs = -1,
        verbose = 1,
        random_state = 42,
        refit = True
    )

    # Optimise only using the outer training data
    optuna_search.fit(X_train, y_train)

    # Best hyperparameters found using inner CV
    best_C = optuna_search.best_params_[
        "classifier__C"
    ]

    best_l1_ratio = optuna_search.best_params_[
        "classifier__l1_ratio"
    ]

    # Predict the completely unseen outer test set
    y_pred = optuna_search.predict(X_test)

    # Probability of positive class for AUC
    y_prob = optuna_search.predict_proba(X_test)[:, 1]

    # Metrics
    mcc = matthews_corrcoef(y_test, y_pred)

    scaled_mcc = (mcc + 1) / 2

    auc = roc_auc_score(y_test, y_prob)

    f1_pos = f1_score(
        y_test,
        y_pred,
        pos_label=1
    )

    f1_neg = f1_score(
        y_test,
        y_pred,
        pos_label=0
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    scores_outer.append({
        "Fold": fold,
        "MCC": mcc,
        "Scaled MCC": scaled_mcc,
        "AUC": auc,
        "F1 pos": f1_pos,
        "F1 neg": f1_neg,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Best C": best_C,
        "Best l1_ratio": best_l1_ratio
    })

# Stopping time
end_time = time.perf_counter()

optimisation_time = end_time - start_time

In [9]:
print(f"{optimisation_time} Seconds")

897.769045699999 Seconds


## Evaluation

In [10]:
metrics_df_outer = pd.DataFrame(scores_outer)

metrics_df_outer

,Fold,MCC,Scaled MCC,AUC,F1 pos,F1 neg,Accuracy,Precision,Recall,Best C,Best l1_ratio
0,1,0.629971,0.814985,0.887882,0.831897,0.797927,0.816471,0.839130,0.824786,1.797466e-01,0.002197
1,2,0.637607,0.818803,0.892335,0.841667,0.794595,0.821176,0.821138,0.863248,3.828300e+02,0.382046
2,3,0.671335,0.835668,0.914194,0.854123,0.816976,0.837647,0.845188,0.863248,4.625615e-01,0.002118
3,4,0.629530,0.814765,0.898868,0.832618,0.796875,0.816471,0.836207,0.829060,3.055874e+00,0.603158
4,5,0.676031,0.838016,0.903387,0.856540,0.819149,0.840000,0.845833,0.867521,6.378929e-01,0.023492
5,6,0.667181,0.833591,0.917662,0.850427,0.816754,0.835294,0.850427,0.850427,3.101255e+04,0.487104
6,7,0.625467,0.812734,0.900143,0.829374,0.795866,0.814118,0.838428,0.820513,1.328845e-01,0.105875
7,8,0.642731,0.821366,0.885354,0.841438,0.801061,0.823529,0.832636,0.850427,9.116110e-02,0.172452
8,9,0.628069,0.814035,0.895735,0.836820,0.790323,0.816471,0.819672,0.854701,1.383308e+01,0.772336
9,10,0.652080,0.826040,0.885846,0.848861,0.801090,0.828235,0.823293,0.876068,2.225457e+00,0.984439


In [11]:
metric_columns = [
    "MCC",
    "Scaled MCC",
    "AUC",
    "F1 pos",
    "F1 neg",
    "Accuracy",
    "Precision",
    "Recall"
]

metrics_df = pd.DataFrame({
    "Mean": metrics_df_outer[metric_columns].mean(),
    "SD": metrics_df_outer[metric_columns].std()
})

metrics_df

,Mean,SD
MCC,0.651142,0.028857
Scaled MCC,0.825571,0.014429
AUC,0.900597,0.014280
F1 pos,0.844533,0.014092
F1 neg,0.805916,0.015182
Accuracy,0.827412,0.014389
Precision,0.837469,0.013234
Recall,0.852137,0.024506


The nested cross-validation took approximately ca. 897 seconds to complete. The loops consisted of 20 outer folds, with each outer fold performing an Optuna hyperparameter optimisation using 50 trials and a repeated stratified 4-fold cross-validation with 25 repeats in the inner loop.

## Questions

Q1: Why do we have 20 rows in dataframe "metrics_df_outer"?  

* Because every row shows the performance metrics for one outer fold. We use 4 splits and 5 repeats and 4 * 5 = 20 outer folds.

<br>

Q2: Why are the optimal hyperparameters different in each row of dataframe "metrics_df_outer"?  

* The optimal hyperparameters can be different because each outer fold uses a different training dataset. Optuna searches for the best C and l1_ratio based on the data in that specific training fold. Since the training data is slightly different for each fold, Optuna can find different optimal hyperparameters.

<br>

Q3: Why does nested cross-validation take longer than ordinary cross-validation?  

* Nested cross-validation takes longer because it has two levels of cross-validation. In the outer loop, the model is evaluated, while in the inner loop Optuna searches for the best hyperparameters. In this assignment, there are 20 outer folds, and for each one there are 50 Optuna trials using 4-fold cross-validation with 25 repeats. This means that many more models have to be trained compared to ordinary cross-validation.

<br>

Q4: How much longer does the nested cross-validation take compared to the cross-validation uses in CA1?  

* The cross-validation in CA1 took 313.37 seconds. The nested cross-validation in CA2 took 897,77. This means that CA2 took about 584,4 seconds longer than CA1.

<br>

Q5: Compare and discuss the cross-validation based performance MCC from CA1 with the nested cross-validation based performance MCC from CA2.  

* The best MCC score in CA1 was 0.654, while the average MCC score from the nested cross-validation in CA2 was 0.651. The two values are very close, with only a small difference of 0.003. This means that the model had a very similar MCC performance in both CA1 and CA2. The CA2 result is based on nested cross-validation and therefore gives a more reliable estimate of the model's performance on unseen data.

<br>

Q6: What do you think about the performance of the model based on nested cross-validation (i.e. CA2 model)? Base your evaluation on the average   performance metrics that you computed and stored in "metrics_df".  

* The average MCC is 0.651, which shows a good agreement between the predicted and actual classes. The average AUC is 0.901, which means that the model is quite good at separating the two classes.  
  
* The average F1 score is 0.845 for the positive class and 0.806 for the negative class. The accuracy is 0.827, meaning that around 83% of the predictions were correct. The precision is 0.837 and the recall is 0.852, which are also high values.  
  
* The standard deviations are relatively small for most of the metrics. For example, the SD for MCC is 0.028, and the SD for AUC is 0.014. This suggests that the model's performance does not change very much between the different outer folds.  

## AI Declaration

I used the AI tool ChatGPT to help improve the formulations of my answers to the questions Q1 - Q2. I also used the AI tool to help understand the Optuna documentation and the descriptions of the hyperparameters used in the optimisation process. The interpretation of the results are my own.